# 04 — BERT Cased: Casing Strategy Ablation
Train `bert-base-cased` with identical config to the uncased baseline and evaluate on clean + noisy data.

In [ ]:
import sys, os, glob
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, load_from_disk
from transformers import (
    AutoTokenizer,
    BertForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
from train import tokenize_and_align_labels, make_compute_metrics

if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
print(f"Device: {DEVICE}")

MODEL_NAME = "bert-base-cased"
RESULTS_DIR = os.path.join(os.path.dirname(os.getcwd()), "results")
TABLES_DIR = os.path.join(RESULTS_DIR, "tables")
CHECKPOINTS_DIR = os.path.join(RESULTS_DIR, "checkpoints")
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data", "noisy")
os.makedirs(TABLES_DIR, exist_ok=True)

In [ ]:
dataset = load_dataset("conll2003", trust_remote_code=True)
label_names = dataset["train"].features["ner_tags"].feature.names
id2label = {i: l for i, l in enumerate(label_names)}
label2id = {l: i for i, l in enumerate(label_names)}
num_labels = len(label_names)
print(f"Labels ({num_labels}): {label_names}")
print(f"Train: {len(dataset['train'])} | Val: {len(dataset['validation'])} | Test: {len(dataset['test'])}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, add_prefix_space=True)
model = BertForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)
print(f"Loaded {MODEL_NAME}")

In [ ]:
tokenized = dataset.map(
    lambda ex: tokenize_and_align_labels(ex, tokenizer),
    batched=True,
    remove_columns=dataset["train"].column_names,
)
print("Tokenized. Example token count:", len(tokenized["train"][0]["input_ids"]))

In [ ]:
training_args = TrainingArguments(
    output_dir=os.path.join(CHECKPOINTS_DIR, MODEL_NAME.replace("/", "_")),
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=100,
    fp16=(DEVICE == "cuda"),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=make_compute_metrics(label_names),
)

trainer.train()

## Evaluate on clean test set

In [ ]:
trainer.callback_handler.on_train_begin(trainer.args, trainer.state, trainer.control)
trainer.eval_dataset = tokenized["test"]
clean_results = trainer.evaluate()
print(f"BERT-cased (clean) | F1: {clean_results['eval_f1']:.4f} | "
      f"Precision: {clean_results['eval_precision']:.4f} | "
      f"Recall: {clean_results['eval_recall']:.4f}")

## Evaluate on noisy test sets

In [ ]:
def evaluate_on_noisy(noise_type):
    noisy_ds = load_from_disk(os.path.join(DATA_DIR, noise_type))
    tokenized_noisy = noisy_ds["test"].map(
        lambda ex: tokenize_and_align_labels(ex, tokenizer),
        batched=True,
        remove_columns=noisy_ds["test"].column_names,
    )
    trainer.callback_handler.on_train_begin(trainer.args, trainer.state, trainer.control)
    trainer.eval_dataset = tokenized_noisy
    metrics = trainer.evaluate()
    return {
        "model": MODEL_NAME,
        "noise": noise_type,
        "f1": round(metrics["eval_f1"], 4),
        "precision": round(metrics["eval_precision"], 4),
        "recall": round(metrics["eval_recall"], 4),
    }

In [ ]:
noisy_results = []
for noise_type in ["ocr", "asr", "social"]:
    print(f"Evaluating on {noise_type}...")
    noisy_results.append(evaluate_on_noisy(noise_type))

noisy_df = pd.DataFrame(noisy_results)
print(noisy_df)

## Summary: Clean + Noisy

In [ ]:
# Baseline from notebook 01 (bert-base-uncased)
uncased_baseline = {
    "ocr": 0.6632, "asr": 0.8255, "social": 0.6174, "clean": 0.890836
}

rows = [{"noise": "clean", "f1_cased": clean_results["eval_f1"],
         "f1_uncased": uncased_baseline["clean"]}]
for r in noisy_results:
    rows.append({
        "noise": r["noise"],
        "f1_cased": r["f1"],
        "f1_uncased": uncased_baseline[r["noise"]],
    })

summary_df = pd.DataFrame(rows)
summary_df["delta_cased_vs_uncased"] = (summary_df["f1_cased"] - summary_df["f1_uncased"]).round(4)
print(summary_df.to_string(index=False))

In [ ]:
# Save noisy results
noisy_df["f1_baseline"] = noisy_df["noise"].map({
    "ocr": clean_results["eval_f1"],
    "asr": clean_results["eval_f1"],
    "social": clean_results["eval_f1"],
})
noisy_df["f1_delta"] = (noisy_df["f1"] - noisy_df["f1_baseline"]).round(4)
noisy_df["f1_drop_pct"] = (noisy_df["f1_delta"] / noisy_df["f1_baseline"] * 100).round(2)

noisy_df.to_csv(os.path.join(TABLES_DIR, "bert_cased_noisy_results.csv"), index=False)

# Save clean result
clean_df = pd.DataFrame([{
    "model": MODEL_NAME,
    "strategy": "WordPiece (cased)",
    "f1_clean": clean_results["eval_f1"],
    "precision_clean": clean_results["eval_precision"],
    "recall_clean": clean_results["eval_recall"],
}])
clean_df.to_csv(os.path.join(TABLES_DIR, "bert_cased_clean_results.csv"), index=False)

print("Saved bert_cased_noisy_results.csv and bert_cased_clean_results.csv")
noisy_df